# 6.4 Other Frameworks → ONNX — Deep Dive

## Table of Contents
1. [Ecosystem Map](#section-1)
2. [XGBoost Tree Ensemble Encoding](#section-2)
3. [LightGBM Leaf Value Mapping](#section-3)
4. [onnxmltools Conversion](#section-4)
5. [Hummingbird — Tensorized Compilation](#section-5)
6. [PaddlePaddle → paddle2onnx](#section-6)
7. [Apache MXNet Export](#section-7)
8. [MATLAB exportONNXNetwork](#section-8)
9. [Converter Comparison Table](#section-9)
10. [Cross-Cutting Validation Playbook](#section-10)
11. [Key Takeaways](#section-11)

<a id='section-1'></a>
## Section 1: Ecosystem Map

The ONNX ecosystem extends far beyond PyTorch and TensorFlow. Every major
ML framework has at least one path to ONNX, though the converter maturity
and maintenance levels vary significantly.

```
┌──────────────────────────────────────────────────────────────────────────┐
│                    FRAMEWORK → ONNX ECOSYSTEM MAP                       │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  ┌─────────────┐                         ┌───────────┐                  │
│  │  PyTorch     │──── torch.onnx.export ──▶│           │                  │
│  └─────────────┘                         │           │                  │
│  ┌─────────────┐                         │           │                  │
│  │ TensorFlow   │──── tf2onnx ───────────▶│           │                  │
│  └─────────────┘                         │           │                  │
│  ┌─────────────┐                         │   ONNX    │  ┌────────────┐ │
│  │ scikit-learn │──── skl2onnx ──────────▶│   Model   │──▶│ ORT / other│ │
│  └─────────────┘                         │  (.onnx)  │  │ runtimes   │ │
│  ┌─────────────┐                         │           │  └────────────┘ │
│  │  XGBoost     │──── onnxmltools ───────▶│           │                  │
│  └─────────────┘     or hummingbird      │           │                  │
│  ┌─────────────┐                         │           │                  │
│  │  LightGBM    │──── onnxmltools ───────▶│           │                  │
│  └─────────────┘     or hummingbird      │           │                  │
│  ┌─────────────┐                         │           │                  │
│  │ PaddlePaddle │──── paddle2onnx ───────▶│           │                  │
│  └─────────────┘                         │           │                  │
│  ┌─────────────┐                         │           │                  │
│  │  MXNet       │──── mxnet.onnx ────────▶│           │                  │
│  └─────────────┘                         │           │                  │
│  ┌─────────────┐                         │           │                  │
│  │  MATLAB      │──── exportONNXNetwork ──▶│           │                  │
│  └─────────────┘                         └───────────┘                  │
│                                                                          │
│  Converter Maturity: ■■■■■ (PyTorch, TF) ■■■■░ (sklearn, XGBoost)      │
│                      ■■■░░ (LightGBM, Paddle) ■■░░░ (MXNet, MATLAB)    │
└──────────────────────────────────────────────────────────────────────────┘
```

### Converter Categories

| Category | Frameworks | Approach |
|----------|-----------|----------|
| **Graph translation** | PyTorch, TF, Paddle, MXNet | Map framework graph nodes to ONNX nodes |
| **Topology reimplementation** | sklearn, XGBoost, LightGBM | Extract parameters, build new ONNX graph |
| **Tensorized compilation** | Any (via Hummingbird) | Compile model into tensor operations, then export |

<a id='section-2'></a>
## Section 2: XGBoost Tree Ensemble Encoding

### XGBoost's Gradient Boosting Model

XGBoost builds an additive model of decision trees using gradient boosting.
Given $M$ boosting rounds with learning rate $\eta$:

$$F(x) = F_0 + \sum_{m=1}^{M} \eta \cdot f_m(x)$$

where $F_0$ is the base prediction (e.g., log-odds of the positive class
for binary classification) and each $f_m$ is a regression tree fit to the
**negative gradient** of the loss at round $m$.

### How XGBoost Trees Differ from sklearn Trees

```
┌─────────────────────────────────────────────────────────────────┐
│         XGBoost vs sklearn TREE DIFFERENCES                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  XGBoost:                                                       │
│  • Leaf values are raw GRADIENT scores (not probabilities)      │
│  • Final prediction: sigmoid/softmax applied AFTER summing      │
│  • Regularization (λ, α) affects leaf values during training   │
│  • Missing value handling built into tree splits                │
│  • Uses learning rate (η) to scale each tree's contribution    │
│                                                                 │
│  sklearn RandomForest:                                          │
│  • Leaf values are CLASS PROBABILITIES (already normalized)     │
│  • Final prediction: average across trees                       │
│  • No learning rate — each tree votes equally                   │
│  • Missing values not natively supported                        │
└─────────────────────────────────────────────────────────────────┘
```

### ONNX Encoding

XGBoost trees are encoded using the `TreeEnsembleRegressor` (for the raw
scores) or `TreeEnsembleClassifier` operator from the `ai.onnx.ml` domain.

Key encoding details:

$$\text{TreeEnsembleRegressor}(x) = \underbrace{F_0}_{\text{base\_values}} + \sum_{m=1}^{M} \underbrace{\eta \cdot \text{leaf\_value}(f_m, x)}_{\text{target\_weights} \times \text{learning\_rate}}$$

The ONNX `TreeEnsembleRegressor` attributes:

| Attribute | Maps to | XGBoost Source |
|-----------|---------|---------------|
| `aggregate_function` | `SUM` | Boosting = additive |
| `base_values` | $F_0$ | Base score (bias) |
| `nodes_featureids` | Split feature index | `feature` in tree dump |
| `nodes_values` | Split threshold | `split_condition` |
| `nodes_missing_value_tracks_true` | Missing → left/right | `missing` direction |
| `target_weights` | $\eta \cdot v_l$ | Leaf values × learning rate |

### Binary Classification Pipeline

For binary classification, the full inference pipeline in ONNX is:

$$P(y=1|x) = \sigma\left(F_0 + \sum_{m=1}^{M} \eta \cdot f_m(x)\right)$$

where $\sigma(z) = \frac{1}{1 + e^{-z}}$ is the sigmoid function.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification, make_regression
import onnx
from onnx import checker

try:
    import xgboost as xgb
    HAS_XGB = True
    print(f"XGBoost version: {xgb.__version__}")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed. Install with: pip install xgboost")

try:
    from onnxmltools import convert_xgboost, convert_lightgbm
    from skl2onnx.common.data_types import FloatTensorType
    HAS_ONNXMLTOOLS = True
    print("onnxmltools available!")
except ImportError:
    HAS_ONNXMLTOOLS = False
    print("onnxmltools not installed. Install with: pip install onnxmltools")

try:
    import onnxruntime as ort
    HAS_ORT = True
except ImportError:
    HAS_ORT = False
    print("onnxruntime not installed. Install with: pip install onnxruntime")

In [ ]:
if HAS_XGB and HAS_ONNXMLTOOLS:
    X_xgb, y_xgb = make_classification(
        n_samples=500, n_features=10, n_informative=7,
        n_classes=2, random_state=42
    )
    X_xgb = X_xgb.astype(np.float32)

    dtrain = xgb.DMatrix(X_xgb, label=y_xgb)
    params = {
        "max_depth": 4,
        "eta": 0.1,
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "nthread": 1,
    }
    bst = xgb.train(params, dtrain, num_boost_round=50)

    initial_type = [("X", FloatTensorType([None, 10]))]
    onnx_xgb = convert_xgboost(bst, initial_types=initial_type, target_opset=17)
    checker.check_model(onnx_xgb)

    print(f"XGBoost model converted!")
    print(f"  Boosting rounds: 50")
    print(f"  Learning rate (eta): 0.1")
    print(f"  ONNX graph nodes: {len(onnx_xgb.graph.node)}")
    print(f"  ONNX ops: {[n.op_type for n in onnx_xgb.graph.node]}")

    serialized = onnx_xgb.SerializeToString()
    print(f"  Model size: {len(serialized):,} bytes ({len(serialized)/1024:.1f} KB)")
else:
    print("Skipping XGBoost example — missing dependencies.")

In [ ]:
if HAS_XGB and HAS_ONNXMLTOOLS and HAS_ORT:
    sess = ort.InferenceSession(
        onnx_xgb.SerializeToString(), providers=["CPUExecutionProvider"]
    )

    X_test = X_xgb[:20]
    dtest = xgb.DMatrix(X_test)
    xgb_proba = bst.predict(dtest)

    ort_input = {sess.get_inputs()[0].name: X_test}
    ort_outputs = sess.run(None, ort_input)

    ort_proba = None
    for out in ort_outputs:
        if isinstance(out, np.ndarray) and out.ndim == 2:
            ort_proba = out[:, 1]
            break
        elif isinstance(out, np.ndarray) and out.ndim == 1 and out.shape[0] == 20:
            ort_proba = out
            break

    if ort_proba is not None:
        diff = np.abs(xgb_proba - ort_proba).max()
        print(f"XGBoost Parity Check:")
        print(f"  Max abs diff: {diff:.2e}")
        print(f"  Parity OK: {diff < 1e-5}")
        print(f"  First 5 XGBoost: {xgb_proba[:5].round(4)}")
        print(f"  First 5 ORT:     {ort_proba[:5].round(4)}")
    else:
        print("Could not extract comparable probability output.")
else:
    print("Skipping parity check — missing dependencies.")

<a id='section-3'></a>
## Section 3: LightGBM Leaf Value Mapping

### LightGBM's Unique Approach

LightGBM (Light Gradient Boosting Machine) differs from XGBoost in several
ways that affect ONNX conversion:

1. **Histogram-based splitting**: LightGBM bins continuous features into
   discrete histograms before finding splits, making splits on bin boundaries
2. **Leaf-wise growth**: Instead of level-wise (like XGBoost default), LGBM
   grows the leaf with the largest loss reduction
3. **Categorical feature handling**: Native support without one-hot encoding

### Boosted Tree Prediction

Like XGBoost, LightGBM builds an additive ensemble:

$$\text{score}(x) = \sum_{m=1}^{M} \text{leaf\_value}(f_m(x))$$

The learning rate is typically absorbed into the leaf values during training,
so the ONNX encoding uses direct leaf values.

### ONNX Encoding Differences from XGBoost

```
┌─────────────────────────────────────────────────────────────────┐
│        LightGBM → ONNX ENCODING                                │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Key Differences from XGBoost encoding:                         │
│                                                                 │
│  1. Histogram bin boundaries → split thresholds                 │
│     • LGBM uses <= comparisons on binned values                 │
│     • ONNX TreeEnsemble uses raw feature thresholds             │
│     • Converter maps bin boundaries back to original scale      │
│                                                                 │
│  2. Categorical splits                                          │
│     • LGBM: native categorical splits (bitmap encoding)         │
│     • ONNX: no native categorical in TreeEnsemble               │
│     • Converter may decompose into multiple binary splits       │
│                                                                 │
│  3. Leaf value scale                                             │
│     • LGBM absorbs learning rate into leaf values                │
│     • ONNX: leaf weights = raw LGBM leaf values (no rescaling)  │
│                                                                 │
│  4. Missing value default direction                              │
│     • Both LGBM and XGBoost support this                        │
│     • Mapped to nodes_missing_value_tracks_true attribute       │
└─────────────────────────────────────────────────────────────────┘
```

### Multi-class LightGBM

For $K$-class classification, LightGBM builds $K$ trees per round (one per
class), producing $K$ raw scores. The final probabilities are:

$$P(y=k|x) = \frac{e^{s_k(x)}}{\sum_{j=1}^{K} e^{s_j(x)}}, \quad s_k(x) = \sum_{m=1}^{M} f_{m,k}(x)$$

In [ ]:
try:
    import lightgbm as lgb
    HAS_LGBM = True
    print(f"LightGBM version: {lgb.__version__}")
except ImportError:
    HAS_LGBM = False
    print("LightGBM not installed. Install with: pip install lightgbm")

if HAS_LGBM and HAS_ONNXMLTOOLS:
    X_lgb, y_lgb = make_classification(
        n_samples=500, n_features=10, n_informative=7,
        n_classes=2, random_state=42
    )
    X_lgb = X_lgb.astype(np.float32)

    dtrain = lgb.Dataset(X_lgb, label=y_lgb)
    params = {
        "objective": "binary",
        "num_leaves": 31,
        "learning_rate": 0.1,
        "verbose": -1,
    }
    lgb_model = lgb.train(params, dtrain, num_boost_round=50)

    initial_type = [("X", FloatTensorType([None, 10]))]
    onnx_lgb = convert_lightgbm(lgb_model, initial_types=initial_type, target_opset=17)
    checker.check_model(onnx_lgb)

    print(f"\nLightGBM model converted!")
    print(f"  Iterations: 50")
    print(f"  ONNX graph nodes: {len(onnx_lgb.graph.node)}")
    print(f"  ONNX ops: {[n.op_type for n in onnx_lgb.graph.node]}")

    serialized = onnx_lgb.SerializeToString()
    print(f"  Model size: {len(serialized):,} bytes ({len(serialized)/1024:.1f} KB)")

    if HAS_ORT:
        sess = ort.InferenceSession(
            serialized, providers=["CPUExecutionProvider"]
        )
        X_test = X_lgb[:20]
        lgb_proba = lgb_model.predict(X_test)

        ort_out = sess.run(None, {sess.get_inputs()[0].name: X_test})
        ort_proba = None
        for out in ort_out:
            if isinstance(out, np.ndarray) and out.ndim == 2:
                ort_proba = out[:, 1]
                break
            elif isinstance(out, np.ndarray) and out.ndim == 1 and out.shape[0] == 20:
                ort_proba = out
                break

        if ort_proba is not None:
            diff = np.abs(lgb_proba - ort_proba).max()
            print(f"\n  Parity check:")
            print(f"    Max abs diff: {diff:.2e}")
            print(f"    Parity OK: {diff < 1e-5}")

elif HAS_LGBM:
    print("onnxmltools not available — cannot convert LightGBM.")
else:
    print("Skipping LightGBM example — missing dependencies.")

<a id='section-4'></a>
## Section 4: onnxmltools Conversion

### What is onnxmltools?

`onnxmltools` is a Python package that provides ONNX converters for various
ML libraries. It serves as a unified conversion interface for frameworks that
don't have their own built-in ONNX exporters.

### Supported Frameworks

| Framework | onnxmltools Function | Dependencies |
|-----------|---------------------|-------------|
| XGBoost | `convert_xgboost()` | xgboost, skl2onnx |
| LightGBM | `convert_lightgbm()` | lightgbm, skl2onnx |
| sklearn | Delegates to `skl2onnx` | skl2onnx |
| CoreML | `convert_coreml()` | coremltools |
| LibSVM | `convert_libsvm()` | libsvm |

### Common API Pattern

All onnxmltools converters follow the same pattern:

```python
from onnxmltools import convert_<framework>
from skl2onnx.common.data_types import FloatTensorType

initial_type = [("X", FloatTensorType([None, n_features]))]
onnx_model = convert_<framework>(
    model,
    initial_types=initial_type,
    target_opset=17
)
```

### Under the Hood

onnxmltools wraps `skl2onnx`'s conversion infrastructure. The process:

```
┌──────────────────────────────────────────────────────────┐
│           onnxmltools CONVERSION PIPELINE                │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  1. Register framework-specific converters               │
│     (XGBoost tree parser, LGBM tree parser, etc.)        │
│                                                          │
│  2. Parse model topology                                 │
│     - Extract tree structures / model parameters         │
│     - Determine input/output shapes                      │
│                                                          │
│  3. Build ONNX graph using skl2onnx infrastructure       │
│     - Create TreeEnsemble nodes                          │
│     - Add post-processing (sigmoid, softmax)             │
│     - Set up initializers for constants                   │
│                                                          │
│  4. Return ModelProto                                     │
└──────────────────────────────────────────────────────────┘
```

<a id='section-5'></a>
## Section 5: Hummingbird — Tensorized Compilation of Trees

### A Different Approach

While onnxmltools maps tree structures directly to ONNX's `TreeEnsemble`
operators, **Hummingbird** takes a radically different approach: it compiles
tree-based models into **tensor computations**.

### The Key Idea

A decision tree can be represented as a series of tensor operations:

```
┌──────────────────────────────────────────────────────────────┐
│           HUMMINGBIRD TENSORIZATION                          │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  Traditional tree traversal:    Tensorized version:          │
│  ─────────────────────────     ─────────────────────         │
│                                                              │
│  if x[0] <= 0.5:               # All splits at once         │
│    if x[1] <= 0.3:             comparisons = (X @ S) <= T   │
│      return A                  # Navigate to leaf            │
│    else:                       leaf_idx = comparisons @ M    │
│      return B                  # Gather predictions          │
│  else:                         preds = gather(V, leaf_idx)   │
│    return C                                                  │
│                                S = selector matrix           │
│  Sequential, branching         T = threshold vector          │
│  (hard to parallelize)         M = navigation matrix         │
│                                V = leaf value vector          │
│                                                              │
│                                Parallel, vectorized           │
│                                (GPU-friendly!)                │
└──────────────────────────────────────────────────────────────┘
```

### Three Compilation Strategies

Hummingbird offers three backend strategies:

| Strategy | Description | Best For |
|----------|------------|----------|
| **GEMM** | All splits as one large matrix multiply | Small trees, GPU |
| **TreeTrav** | Tree traversal via tensor indexing | Deep trees |
| **PerfTreeTrav** | Optimized traversal with bit manipulation | Very large ensembles |

### When to Use Hummingbird vs onnxmltools

| Criterion | onnxmltools | Hummingbird |
|-----------|------------|------------|
| CPU inference | Faster (native tree ops) | Slower (tensor overhead) |
| GPU inference | N/A (TreeEnsemble is CPU-only) | **Much faster** |
| Model fidelity | Exact tree reproduction | Exact (mathematically equivalent) |
| Dependencies | Minimal | PyTorch (as compilation backend) |
| ONNX graph style | `ai.onnx.ml` domain ops | Standard tensor ops |

In [ ]:
try:
    from hummingbird.ml import convert as hb_convert
    HAS_HUMMINGBIRD = True
    print("Hummingbird available!")
except ImportError:
    HAS_HUMMINGBIRD = False
    print("Hummingbird not installed. Install with: pip install hummingbird-ml")

if HAS_HUMMINGBIRD and HAS_XGB:
    from sklearn.ensemble import RandomForestClassifier

    X_hb, y_hb = make_classification(
        n_samples=300, n_features=8, random_state=42
    )
    X_hb = X_hb.astype(np.float32)

    rf = RandomForestClassifier(n_estimators=20, max_depth=5, random_state=0)
    rf.fit(X_hb, y_hb)

    hb_model = hb_convert(rf, "torch")

    sk_preds = rf.predict(X_hb[:10])
    hb_preds = hb_model.predict(X_hb[:10])

    print(f"Hummingbird compilation successful!")
    print(f"  sklearn predictions: {sk_preds}")
    print(f"  Hummingbird preds:   {hb_preds}")
    print(f"  Match: {np.array_equal(sk_preds, hb_preds)}")
else:
    print("Skipping Hummingbird example — missing dependencies.")

<a id='section-6'></a>
## Section 6: PaddlePaddle → paddle2onnx

### Overview

PaddlePaddle (Paddle) is Baidu's deep learning framework, popular in China
and increasingly used globally. `paddle2onnx` converts Paddle models (both
static and dynamic graph modes) to ONNX.

### Conversion Paths

```
┌──────────────────────────────────────────────────────────┐
│           PaddlePaddle → ONNX                            │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  Static Graph (saved model):                             │
│  ┌──────────────┐                                        │
│  │ model.pdmodel │──┐                                    │
│  │ model.pdiparams│──┤── paddle2onnx ──▶ model.onnx     │
│  └──────────────┘  │                                     │
│                                                          │
│  Dynamic Graph (Layer subclass):                         │
│  ┌──────────────┐                                        │
│  │ paddle.jit    │                                       │
│  │ .save()       │── export to static ──┐                │
│  └──────────────┘                       │                │
│                                         ├── paddle2onnx  │
│                                         │   ──▶ .onnx    │
│                                         │                │
│  Python API:                                              │
│  paddle2onnx.export(model_file, params_file, save_file)  │
└──────────────────────────────────────────────────────────┘
```

### CLI Usage

```bash
pip install paddle2onnx

# From saved inference model
paddle2onnx \
    --model_dir ./inference_model \
    --model_filename model.pdmodel \
    --params_filename model.pdiparams \
    --save_file model.onnx \
    --opset_version 17 \
    --enable_onnx_checker True
```

### Python API

```python
import paddle2onnx

paddle2onnx.export(
    model_file="model.pdmodel",
    params_file="model.pdiparams",
    save_file="model.onnx",
    opset_version=17,
    enable_onnx_checker=True,
)
```

### Supported Model Zoo

paddle2onnx has been tested with Paddle's major model libraries:

| Library | Models | Status |
|---------|--------|--------|
| PaddleDetection | YOLO, RCNN, SSD | Good |
| PaddleOCR | CRNN, DBNet | Good |
| PaddleClas | ResNet, EfficientNet | Good |
| PaddleNLP | ERNIE, BERT | Partial |
| PaddleSeg | U-Net, DeepLab | Good |

<a id='section-7'></a>
## Section 7: Apache MXNet Export

### Overview

Apache MXNet provides ONNX export capabilities for its symbolic and Gluon
APIs. However, MXNet's community focus has shifted (the project is now in
the Apache Attic), so the ONNX exporter may not support the latest ONNX
opsets.

### Export Methods

```python
# Method 1: From Symbol API (older MXNet)
import mxnet as mx
from mxnet.contrib import onnx as onnx_mxnet

onnx_mxnet.export_model(
    sym="model-symbol.json",
    params="model-0000.params",
    input_shape=[(1, 3, 224, 224)],
    onnx_file_path="model.onnx"
)

# Method 2: From hybridized Gluon block
net.hybridize()
net(mx.nd.zeros((1, 3, 224, 224)))
net.export("model")
# Then use export_model on the exported files
```

### Considerations

| Aspect | Details |
|--------|---------|
| Opset support | Typically opset 12-14 (may lag latest) |
| Operator coverage | Core DL ops well-supported |
| Maintenance | Limited (MXNet in Apache Attic) |
| Alternative | Re-train in PyTorch/TF if possible |

<a id='section-8'></a>
## Section 8: MATLAB exportONNXNetwork

### Overview

MathWorks provides `exportONNXNetwork` as part of the Deep Learning Toolbox.
This enables MATLAB-trained neural networks to be deployed via ONNX Runtime
or other ONNX-compatible runtimes.

### MATLAB API

```matlab
% Basic export
exportONNXNetwork(net, 'model.onnx')

% With opset specification
exportONNXNetwork(net, 'model.onnx', 'OpsetVersion', 13)
```

### Supported Layer Types

```
┌──────────────────────────────────────────────────────────┐
│           MATLAB → ONNX LAYER SUPPORT                    │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  Well-Supported:                                         │
│  • convolution2dLayer → Conv                             │
│  • fullyConnectedLayer → Gemm                            │
│  • reluLayer → Relu                                      │
│  • batchNormalizationLayer → BatchNormalization           │
│  • maxPooling2dLayer → MaxPool                            │
│  • averagePooling2dLayer → AveragePool                   │
│  • softmaxLayer → Softmax                                │
│  • lstmLayer → LSTM                                      │
│                                                          │
│  Limited / Unsupported:                                  │
│  • Custom layers (must define ONNX mapping)              │
│  • Some attention mechanisms                             │
│  • MATLAB-specific preprocessing layers                  │
└──────────────────────────────────────────────────────────┘
```

### Typical Workflow

1. Train network in MATLAB using Deep Learning Toolbox
2. Export with `exportONNXNetwork(net, 'model.onnx')`
3. Validate in Python with `onnx.checker.check_model()`
4. Deploy with ONNX Runtime

This path is particularly common in engineering organizations where
R&D uses MATLAB but production runs on Python/C++ servers.

### XGBoost vs LightGBM ONNX Size Comparison

An important practical consideration is the ONNX model size. Tree
ensembles can produce surprisingly large ONNX files because every
split condition is stored explicitly.

The model size scales roughly as:

$$\text{size} \propto M \cdot \bar{N}_{\text{nodes}} \cdot (\text{sizeof}(\text{feature\_id}) + \text{sizeof}(\text{threshold}) + \text{sizeof}(\text{mode}))$$

where $M$ is the number of trees and $\bar{N}_{\text{nodes}}$ is the
average number of nodes per tree.

In [ ]:
print("Model Size Comparison (if available):")
print(f"{'Model':<25} {'ONNX Size':>12} {'Trees':>8} {'Notes':>20}")
print("-" * 70)

if HAS_XGB and HAS_ONNXMLTOOLS:
    xgb_size = len(onnx_xgb.SerializeToString())
    print(f"{'XGBoost (50 rounds)':<25} {xgb_size:>10,} B {50:>8} {'binary, depth=4':>20}")

if HAS_LGBM and HAS_ONNXMLTOOLS:
    lgb_size = len(onnx_lgb.SerializeToString())
    print(f"{'LightGBM (50 rounds)':<25} {lgb_size:>10,} B {50:>8} {'binary, 31 leaves':>20}")

if not (HAS_XGB and HAS_ONNXMLTOOLS) and not (HAS_LGBM and HAS_ONNXMLTOOLS):
    print("  No converted models available for comparison.")
    print("  Typical sizes: 50-tree XGBoost ≈ 50-200 KB, 50-tree LGBM ≈ 50-300 KB")

<a id='section-9'></a>
## Section 9: Converter Comparison Table

### When to Use Which Converter

```
┌──────────────────────────────────────────────────────────────────────────┐
│                    CONVERTER DECISION MATRIX                             │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  What framework did you train in?                                        │
│  ┌────────────────────────────────────────────┐                          │
│  │ PyTorch      → torch.onnx.export           │                          │
│  │ TensorFlow   → tf2onnx                     │                          │
│  │ sklearn      → skl2onnx                    │                          │
│  │ XGBoost      → onnxmltools                 │                          │
│  │ LightGBM     → onnxmltools                 │                          │
│  │ PaddlePaddle → paddle2onnx                 │                          │
│  │ MXNet        → mxnet.contrib.onnx          │                          │
│  │ MATLAB       → exportONNXNetwork            │                          │
│  └────────────────────────────────────────────┘                          │
│                                                                          │
│  Do you need GPU inference for tree models?                              │
│  ┌────────────────────────────────────────────┐                          │
│  │ YES  → Hummingbird (tensorized trees)      │                          │
│  │ NO   → onnxmltools (native tree ONNX)      │                          │
│  └────────────────────────────────────────────┘                          │
└──────────────────────────────────────────────────────────────────────────┘
```

### Comprehensive Comparison

| Source | Converter | Maturity | Output Style | Opset Range | GPU-Friendly |
|--------|-----------|----------|-------------|-------------|-------------|
| PyTorch | torch.onnx.export | High | Standard tensor ONNX | 7-18+ | Yes |
| TensorFlow | tf2onnx | High | TF→ONNX node mapping | 7-18+ | Yes |
| scikit-learn | skl2onnx | High | ai.onnx.ml + tensor | 7-17+ | Partial |
| XGBoost | onnxmltools | Medium | ai.onnx.ml trees | 7-17 | No (TreeEnsemble) |
| LightGBM | onnxmltools | Medium | ai.onnx.ml trees | 7-17 | No (TreeEnsemble) |
| XGB/LGBM/sklearn | hummingbird-ml | Medium | Compiled tensor ops | Depends | Yes |
| PaddlePaddle | paddle2onnx | Medium | Program→ONNX | 7-17+ | Yes |
| MXNet | mxnet.contrib.onnx | Low | Symbol→ONNX | 7-14 | Yes |
| MATLAB | exportONNXNetwork | Medium | DL Toolbox→ONNX | 7-13 | Yes |

In [ ]:
print("Framework → ONNX Converter Installation Commands")
print("=" * 55)
print()

converters = {
    "PyTorch": "pip install torch onnx",
    "TensorFlow": "pip install tensorflow tf2onnx",
    "scikit-learn": "pip install skl2onnx",
    "XGBoost": "pip install xgboost onnxmltools",
    "LightGBM": "pip install lightgbm onnxmltools",
    "Hummingbird": "pip install hummingbird-ml[extra]",
    "PaddlePaddle": "pip install paddlepaddle paddle2onnx",
    "Runtime": "pip install onnxruntime",
}

for framework, cmd in converters.items():
    print(f"  {framework:15s} │ {cmd}")

<a id='section-10'></a>
## Section 10: Cross-Cutting Validation Playbook

Regardless of the source framework, **every** ONNX conversion must pass
the same validation gauntlet. Here is a universal playbook.

### The Three-Layer Validation

```
┌──────────────────────────────────────────────────────────────┐
│           UNIVERSAL ONNX VALIDATION PLAYBOOK                │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  LAYER 1: Structural Validation                              │
│  ─────────────────────────────                               │
│  • onnx.checker.check_model(model)                           │
│  • Verify opset version matches target runtime               │
│  • Check input/output names and shapes                       │
│  • Ensure no unsupported operator domains                    │
│                                                              │
│  LAYER 2: Numerical Validation                               │
│  ──────────────────────────────                               │
│  • Generate N representative test inputs                     │
│  • Run through source framework                              │
│  • Run through ONNX Runtime                                  │
│  • Compare: max|diff| < ε (1e-5 for fp32)                   │
│  • Test edge cases: empty batch, max batch, extreme values   │
│                                                              │
│  LAYER 3: Performance Validation                             │
│  ────────────────────────────────                             │
│  • Measure latency with target Execution Provider             │
│  • Compare throughput vs source framework                    │
│  • Profile memory usage                                      │
│  • Test under production-like load                            │
│                                                              │
└──────────────────────────────────────────────────────────────┘
```

### Tolerance Guidelines by Model Type

| Model Type | Expected Max Diff (fp32) | Notes |
|-----------|-------------------------|-------|
| Linear models | $< 10^{-6}$ | Should be near-exact |
| Tree ensembles | $= 0$ (exact) | Deterministic tree traversal |
| CNNs | $< 10^{-5}$ | Floating-point arithmetic differences |
| Transformers | $< 10^{-4}$ | Attention softmax sensitivity |
| TF→ONNX with layout conversion | $< 10^{-4}$ | Transpose insertion effects |

### CI Integration

A minimal CI parity test:

```python
def test_onnx_parity():
    x = generate_test_input()
    y_source = source_model.predict(x)
    y_onnx = ort_session.run(None, {"input": x})[0]
    np.testing.assert_allclose(y_source, y_onnx, atol=1e-5, rtol=1e-4)
```

In [ ]:
import onnx
from onnx import checker
import numpy as np

def validate_onnx_model(model_proto, test_name="Model"):
    """Universal ONNX validation function."""
    print(f"\n{'=' * 50}")
    print(f"Validating: {test_name}")
    print(f"{'=' * 50}")

    try:
        checker.check_model(model_proto)
        print(f"  [PASS] Structural validation")
    except Exception as e:
        print(f"  [FAIL] Structural validation: {e}")
        return False

    opset = model_proto.opset_import[0].version
    print(f"  Opset version: {opset}")

    num_nodes = len(model_proto.graph.node)
    unique_ops = sorted(set(n.op_type for n in model_proto.graph.node))
    print(f"  Nodes: {num_nodes}")
    print(f"  Unique ops: {unique_ops}")

    for inp in model_proto.graph.input:
        shape = inp.type.tensor_type.shape
        if shape:
            dims = [d.dim_param or str(d.dim_value) for d in shape.dim]
            print(f"  Input '{inp.name}': [{', '.join(dims)}]")

    for out in model_proto.graph.output:
        print(f"  Output: '{out.name}'")

    serialized_size = len(model_proto.SerializeToString())
    print(f"  Serialized size: {serialized_size:,} bytes ({serialized_size/1024:.1f} KB)")

    return True

print("validate_onnx_model() function defined — use for any converted model.")

if HAS_XGB and HAS_ONNXMLTOOLS:
    validate_onnx_model(onnx_xgb, "XGBoost Binary Classifier")

if HAS_LGBM and HAS_ONNXMLTOOLS:
    validate_onnx_model(onnx_lgb, "LightGBM Binary Classifier")

<a id='section-11'></a>
## Section 11: Key Takeaways

### Universal Conversion Principles

```
┌─────────────────────────────────────────────────────────────────┐
│           UNIVERSAL ONNX CONVERSION PRINCIPLES                 │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. CHOOSE THE RIGHT CONVERTER                                 │
│     Every framework has a primary path — use it                 │
│     Consider Hummingbird for GPU tree inference                 │
│                                                                 │
│  2. VALIDATE AT THREE LEVELS                                    │
│     Structural: onnx.checker.check_model()                      │
│     Numerical:  max|diff| < ε on representative inputs         │
│     Performance: measure with target EP                         │
│                                                                 │
│  3. PIN YOUR VERSIONS                                           │
│     Framework version + Converter version + Opset version       │
│     = Reproducible conversion                                   │
│                                                                 │
│  4. INCLUDE PARITY TESTS IN CI                                  │
│     Catch regressions from framework/converter updates          │
│     Test with realistic inputs, not just random noise           │
│                                                                 │
│  5. UNDERSTAND THE CONVERSION APPROACH                          │
│     Graph translation (DL frameworks) vs                        │
│     Topology reimplementation (classical ML) vs                 │
│     Tensorized compilation (Hummingbird)                        │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Core Concepts Summary

| Concept | Key Insight |
|---------|------------|
| **XGBoost encoding** | Boosted trees → TreeEnsembleRegressor with `aggregate_function=SUM`; $F(x) = F_0 + \sum \eta f_m(x)$ |
| **LightGBM encoding** | Histogram-based splits mapped back to raw thresholds; categorical splits decomposed into binary |
| **onnxmltools** | Unified converter for XGBoost, LightGBM, and others; wraps skl2onnx infrastructure |
| **Hummingbird** | Tensorizes tree models for GPU execution; three strategies (GEMM, TreeTrav, PerfTreeTrav) |
| **paddle2onnx** | Converts Paddle static/dynamic graphs; good coverage for Paddle model zoo |
| **Validation playbook** | Three layers: structural → numerical → performance; include in CI |